In [0]:
display(dbutils.fs.ls("s3://claims-pipeline-raws/raw/"))

path,name,size,modificationTime
s3://claims-pipeline-raws/raw/beneficiary/,beneficiary/,0,1785991349475
s3://claims-pipeline-raws/raw/carrier/,carrier/,0,1785991349475
s3://claims-pipeline-raws/raw/inpatient/,inpatient/,0,1785991349475
s3://claims-pipeline-raws/raw/outpatient/,outpatient/,0,1785991349475
s3://claims-pipeline-raws/raw/pde/,pde/,0,1785991349475


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS healthcare_claims_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS healthcare_claims_catalog.bronze;

In [0]:
from pyspark.sql import functions as F

def ingest_to_bronze(file_paths, table_name):
    """
    file_paths : string or list of strings
    table_name : Bronze table name
    """

    if isinstance(file_paths, str):
        file_paths = [file_paths]

    df = None

    for path in file_paths:
        temp = (
            spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(path)
                .select("*", "_metadata")
        )

        if df is None:
            df = temp
        else:
            df = df.unionByName(temp)

    bronze_df = (
        df.withColumn("ingestion_timestamp", F.current_timestamp())
          .withColumn("source_file_name", F.col("_metadata.file_path"))
          .withColumn("file_modification_time", F.col("_metadata.file_modification_time"))
          .drop("_metadata")
    )

    bronze_df.write \
        .format("delta") \
        .mode("append")\
        .option("overwriteSchema", "true") \
        .saveAsTable(f"healthcare_claims_catalog.bronze.{table_name}")

    print(f"{table_name} loaded successfully.")

In [0]:
ingest_to_bronze(
    [
        "s3://claims-pipeline-raws/raw/beneficiary/beneficiary_2008.csv",
        "s3://claims-pipeline-raws/raw/beneficiary/beneficiary_2009.csv",
        "s3://claims-pipeline-raws/raw/beneficiary/beneficiary_2010.csv"
    ],
    "bronze_beneficiary"
)

bronze_beneficiary loaded successfully.


In [0]:
%sql
SHOW TABLES IN healthcare_claims_catalog.bronze;

database,tableName,isTemporary
bronze,bronze_beneficiary,false


In [0]:
%sql
select count(*) from healthcare_claims_catalog.bronze.bronze_beneficiary;

count(*)
343857


In [0]:
ingest_to_bronze(
    [
        "s3://claims-pipeline-raws/raw/carrier/carrier_claimA.csv",
        "s3://claims-pipeline-raws/raw/carrier/carrier_claimB.csv"

    ],
    "bronze_carrier"
)

bronze_carrier loaded successfully.


In [0]:
ingest_to_bronze(
    [
        "s3://claims-pipeline-raws/raw/inpatient/inpatient.csv"

    ],
    "bronze_inpatient"
)

bronze_inpatient loaded successfully.


In [0]:
ingest_to_bronze(
    [
        "s3://claims-pipeline-raws/raw/outpatient/outpatient.csv"

    ],
    "bronze_outpatient"
)

bronze_outpatient loaded successfully.


In [0]:
ingest_to_bronze(
    [
        "s3://claims-pipeline-raws/raw/pde/pde.csv"

    ],
    "bronze_pde"
)

bronze_pde loaded successfully.


In [0]:
%sql
SHOW TABLES IN healthcare_claims_catalog.bronze.bronze_inpatient;

database,tableName,isTemporary
bronze,bronze_beneficiary,false
bronze,bronze_carrier,false
bronze,bronze_inpatient,false
bronze,bronze_outpatient,false
bronze,bronze_pde,false
